In [2]:
import numpy as np

# -----------------------------------------
# STEP 1: Define the Hidden States (Tags)
# -----------------------------------------
tags = ["Noun", "Verb"]

# -----------------------------------------
# STEP 2: Define the Vocabulary
# -----------------------------------------
words = ["cat", "meow"]

# Convert words into indices
word_to_index = {
    "cat": 0,
    "meow": 1
}

# -----------------------------------------
# STEP 3: Initial Probability
# P(Tag at first word)
# -----------------------------------------
init_prob = np.array([
    0.6,  # Noun
    0.4   # Verb
])

# -----------------------------------------
# STEP 4: Transition Probability Matrix
# P(Current Tag | Previous Tag)
#
#         To
#       Noun  Verb
# From
# Noun  0.7   0.3
# Verb  0.4   0.6
# -----------------------------------------
transition = np.array([
    [0.7, 0.3],  # From Noun
    [0.4, 0.6]   # From Verb
])

# -----------------------------------------
# STEP 5: Emission Probability Matrix
# P(Word | Tag)
#
#         cat  meow
# Noun    0.8   0.2
# Verb    0.1   0.9
# -----------------------------------------
emission = np.array([
    [0.8, 0.2],
    [0.1, 0.9]
])

# -----------------------------------------
# STEP 6: Convert probabilities into Log Space
# This avoids numerical underflow.
# -----------------------------------------
log_init = np.log(init_prob + 1e-10)
log_trans = np.log(transition + 1e-10)
log_emit = np.log(emission + 1e-10)

# -----------------------------------------
# STEP 7: Viterbi Algorithm
# -----------------------------------------
def viterbi(sentence):
    # Number of words
    T = len(sentence)

    # Number of tags
    N = len(tags)

    # DP table
    dp = np.full((N, T), -np.inf)

    # Backpointer table
    backpointer = np.zeros((N, T), dtype=int)

    # ---------------- Initialization ----------------
    # Calculate probability for the first word
    dp[:, 0] = log_init + log_emit[:, sentence[0]]

    # ---------------- Recursion ----------------
    # Process remaining words
    for t in range(1, T):
        # Check every current tag
        for j in range(N):
            # Calculate score from every previous tag
            scores = (
                dp[:, t - 1]
                + log_trans[:, j]
                + log_emit[j, sentence[t]]
            )

            # Store best previous tag
            backpointer[j, t] = np.argmax(scores)

            # Store best score
            dp[j, t] = np.max(scores)

    # ---------------- Backtracking ----------------
    # Best tag for last word
    best_path = [np.argmax(dp[:, -1])]

    # Trace backwards
    for t in range(T - 1, 0, -1):
        best_path.append(backpointer[best_path[-1], t])

    # Reverse to get correct order
    best_path.reverse()

    return best_path

# -----------------------------------------
# STEP 8: Example Sentence
# -----------------------------------------
sentence = ["cat", "meow"]

# Convert words into indices
sentence_index = [word_to_index[word] for word in sentence]

# Run Viterbi
best_tags = viterbi(sentence_index)

# -----------------------------------------
# STEP 9: Print Results
# -----------------------------------------
print("Sentence:", sentence)
print("\nPredicted Tags:")

for word, tag in zip(sentence, best_tags):
    print(word, "->", tags[tag])

Sentence: ['cat', 'meow']

Predicted Tags:
cat -> Noun
meow -> Verb
